In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 47. B8 Project — Treasury Predictive Uncertainty and Latent-State Audit

> latent stateは市場の隠れた真実ではない。観測系列を圧縮するmodel componentとして、外部予測と安定性で反証する。

## 学習目標

- B7と同じ5公表日先curve targetとouter testを再利用できる
- Bayesian regression posterior predictiveとHMM conditional predictiveを区別できる
- state数をtraining/validationだけで固定できる
- point RMSE、coverage、width、log score、occupancy、duration、transition stabilityを監査できる
- label switching、parameter uncertainty不足、no-selectionをclaimへ反映できる

## 前提知識

- Week 29–32の全Exit Criteria
- B7 Projectのlocked data/horizon/split

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 47


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Locked contract

| Item | Rule |
|---|---|
| Target | five-publication change of all five Treasury tenors |
| Point baseline | zero change (random walk in levels) |
| Bayesian model | tenor-wise NIG regression posterior predictive |
| Latent model | diagonal-Gaussian HMM on fixed-decay NS factor changes |
| State count | 2/3/4 fit on training, select by validation log score |
| Outer test | unchanged B5/B6/B7 start, one use |
| HMM uncertainty | emission/state simulation conditional on point-estimated parameters |
| Prohibited | state=true regime、full Bayes claim、causality、PnL |

In [4]:
decay = 0.5
loadings = qt.nelson_siegel_loadings(maturity_years, decay)
factors = qt.extract_nelson_siegel_factors(curve_yields, maturity_years, decay)
factor_changes_bp = np.diff(factors, axis=0) * 100.0
factor_change_dates = curve_dates[1:]
factor_training = factor_change_dates <= train_end_date
factor_validation = (factor_change_dates > train_end_date) & (factor_change_dates <= validation_end_date)

selection_rows = []
candidate_models = {}
for state_count in [2, 3, 4]:
    candidate = qt.fit_gaussian_hmm(factor_changes_bp[factor_training], state_count)
    candidate_models[state_count] = candidate
    validation_score = qt.hmm_log_likelihood(candidate, factor_changes_bp[factor_validation])
    selection_rows.append(
        {"states": state_count, "validation_log_score_per_observation": validation_score / np.sum(factor_validation), "converged": candidate.converged}
    )
selection_table = pd.DataFrame(selection_rows)
selected_states = int(selection_table.loc[selection_table["validation_log_score_per_observation"].idxmax(), "states"])
display(selection_table)
print("selected states before outer test:", selected_states)

,states,validation_log_score_per_observation,converged
0,2,-13.062540,True
1,3,-12.351554,True
2,4,-11.923496,True


selected states before outer test: 4


## 2. Pretest refit and filtered state audit

In [5]:
pretest_changes = factor_change_dates < test_start_date
hmm = qt.fit_gaussian_hmm(factor_changes_bp[pretest_changes], selected_states)
filtered_probability = qt.hmm_filtered_probabilities(hmm, factor_changes_bp)
diagnostics = qt.hmm_state_diagnostics(hmm, factor_changes_bp[pretest_changes])
state_table = pd.DataFrame(
    {
        "state": np.arange(selected_states),
        "level_change_mean_bp": hmm.means[:, 0],
        "slope_change_mean_bp": hmm.means[:, 1],
        "curvature_change_mean_bp": hmm.means[:, 2],
        "occupancy": diagnostics.occupancy,
        "mean_duration": diagnostics.mean_duration,
    }
)
display(state_table)

fig = go.Figure()
for state in range(selected_states):
    fig.add_scatter(x=factor_change_dates, y=filtered_probability[:, state], name=f"state {state}", mode="lines")
fig.add_vline(x=pd.Timestamp(test_start_date).timestamp() * 1000, line_dash="dash", line_color="black")
fig.update_layout(title="Online HMM filtered probabilities under pretest parameters", yaxis_title="Probability", template="plotly_white")
fig.show()

,state,level_change_mean_bp,slope_change_mean_bp,curvature_change_mean_bp,occupancy,mean_duration
0,0,-3.438022,3.949243,-0.613710,0.346049,1.561475
1,1,-1.512960,2.653022,-2.528265,0.150318,3.484211
2,2,1.391397,-1.479307,1.343439,0.347411,1.645161
3,3,6.771603,-7.207273,0.930905,0.156222,1.307985


## 3. Same-target Bayesian and HMM predictive distributions

Bayesian regressionはparameterとobservation uncertaintyを積分する。HMM sampleはpoint-estimated transition/emission parametersを固定するためfull posterior predictiveではない。この差を結果表にも残す。

In [6]:
horizon = 5
all_origins = np.arange(curve_yields.shape[0] - horizon)
all_targets_bp = (curve_yields[all_origins + horizon] - curve_yields[all_origins]) * 100.0
all_target_dates = curve_dates[all_origins + horizon]
training_rows = all_target_dates < test_start_date
test_rows = curve_dates[all_origins] >= test_start_date

raw_features = np.column_stack(
    [curve_yields[all_origins], np.vstack([np.zeros(5), np.diff(curve_yields, axis=0)])[all_origins] * 100.0]
)
feature_mean = raw_features[training_rows].mean(axis=0)
feature_scale = raw_features[training_rows].std(axis=0, ddof=1)
design = np.column_stack([np.ones(raw_features.shape[0]), (raw_features - feature_mean) / feature_scale])

bayesian_means = []
bayesian_lowers = []
bayesian_uppers = []
for tenor_index in range(5):
    model = qt.fit_bayesian_linear_regression(
        design[training_rows], all_targets_bp[training_rows, tenor_index], prior_precision=1.0, prior_shape=2.0, prior_scale=25.0
    )
    predictive = qt.bayesian_linear_predictive(model, design[test_rows])
    lower, upper = predictive.interval(0.9)
    bayesian_means.append(predictive.mean)
    bayesian_lowers.append(lower)
    bayesian_uppers.append(upper)
bayesian_mean = np.column_stack(bayesian_means)
bayesian_lower = np.column_stack(bayesian_lowers)
bayesian_upper = np.column_stack(bayesian_uppers)

test_origins = all_origins[test_rows]
hmm_mean = np.empty((test_origins.size, 5))
hmm_lower = np.empty_like(hmm_mean)
hmm_upper = np.empty_like(hmm_mean)
for row, origin in enumerate(test_origins):
    probability = filtered_probability[origin - 1]
    factor_draws = qt.simulate_hmm_forecast(
        hmm,
        probability,
        horizon,
        600,
        rng=task_rng(10, int(origin)),
    ).sum(axis=1)
    curve_draws = factor_draws @ loadings.T
    hmm_mean[row] = curve_draws.mean(axis=0)
    hmm_lower[row] = np.quantile(curve_draws, 0.05, axis=0)
    hmm_upper[row] = np.quantile(curve_draws, 0.95, axis=0)

In [7]:
test_actual = all_targets_bp[test_rows]
evaluation_rows = []
for name, mean, lower, upper, uncertainty in [
    ("Bayesian regression", bayesian_mean, bayesian_lower, bayesian_upper, "posterior predictive"),
    ("HMM", hmm_mean, hmm_lower, hmm_upper, "parameter-conditional predictive"),
]:
    evaluation_rows.append(
        {
            "model": name,
            "uncertainty_contract": uncertainty,
            "aggregate_rmse_bp": np.sqrt(np.mean((test_actual - mean) ** 2)),
            "marginal_coverage_90": np.mean((test_actual >= lower) & (test_actual <= upper)),
            "mean_interval_width_bp": np.mean(upper - lower),
        }
    )
evaluation_rows.insert(
    0,
    {
        "model": "random walk",
        "uncertainty_contract": "point baseline only",
        "aggregate_rmse_bp": np.sqrt(np.mean(test_actual**2)),
        "marginal_coverage_90": np.nan,
        "mean_interval_width_bp": np.nan,
    },
)
evaluation_table = pd.DataFrame(evaluation_rows)
display(evaluation_table)
random_walk_rmse = float(evaluation_table.loc[evaluation_table["model"] == "random walk", "aggregate_rmse_bp"].iloc[0])
adoptable = evaluation_table[
    (evaluation_table["model"] != "random walk")
    & (evaluation_table["aggregate_rmse_bp"] < random_walk_rmse)
    & (evaluation_table["marginal_coverage_90"].between(0.85, 0.95))
]
print("project conclusion:", "no model selected" if adoptable.empty else adoptable["model"].tolist())

tenor_rows = []
for tenor_index, tenor in enumerate(qt.DEFAULT_TENORS):
    tenor_rows.append(
        {
            "tenor": tenor,
            "random_walk_rmse_bp": np.sqrt(np.mean(test_actual[:, tenor_index] ** 2)),
            "bayesian_rmse_bp": np.sqrt(np.mean((test_actual[:, tenor_index] - bayesian_mean[:, tenor_index]) ** 2)),
            "hmm_rmse_bp": np.sqrt(np.mean((test_actual[:, tenor_index] - hmm_mean[:, tenor_index]) ** 2)),
            "bayesian_coverage_90": np.mean((test_actual[:, tenor_index] >= bayesian_lower[:, tenor_index]) & (test_actual[:, tenor_index] <= bayesian_upper[:, tenor_index])),
            "hmm_coverage_90": np.mean((test_actual[:, tenor_index] >= hmm_lower[:, tenor_index]) & (test_actual[:, tenor_index] <= hmm_upper[:, tenor_index])),
        }
    )
display(pd.DataFrame(tenor_rows))

,model,uncertainty_contract,aggregate_rmse_bp,marginal_coverage_90,mean_interval_width_bp
0,random walk,point baseline only,11.133358,NaN,NaN
1,Bayesian regression,posterior predictive,11.260134,0.882288,33.720491
2,HMM,parameter-conditional predictive,11.288678,0.926199,44.355241


project conclusion: no model selected


,tenor,random_walk_rmse_bp,bayesian_rmse_bp,hmm_rmse_bp,bayesian_coverage_90,hmm_coverage_90
0,3m,5.091053,5.750963,5.678171,0.948339,0.987085
1,2y,12.066697,12.119705,12.234650,0.869004,0.926199
2,5y,12.934621,12.991878,13.053995,0.867159,0.926199
3,10y,12.116059,12.167989,12.209285,0.870849,0.900369
4,30y,11.581494,11.710825,11.652018,0.856089,0.891144


## 4. Transition stability and label sensitivity

In [8]:
pretest_values = factor_changes_bp[pretest_changes]
split_point = pretest_values.shape[0] // 2
early = qt.fit_gaussian_hmm(pretest_values[:split_point], selected_states)
late = qt.fit_gaussian_hmm(pretest_values[split_point:], selected_states)
stability_table = pd.DataFrame(
    [
        {
            "diagnostic": "transition Frobenius distance",
            "value": np.linalg.norm(early.transition_matrix - late.transition_matrix),
        },
        {
            "diagnostic": "emission-mean Frobenius distance",
            "value": np.linalg.norm(early.means - late.means),
        },
        {
            "diagnostic": "minimum full-pretest occupancy",
            "value": diagnostics.occupancy.min(),
        },
        {
            "diagnostic": "maximum classical state duration",
            "value": diagnostics.mean_duration.max(),
        },
    ]
)
display(stability_table)
print("labels canonicalized by level-factor change mean:", True)
print("labels are external observed regimes:", False)
print("HMM parameter posterior included:", False)

,diagnostic,value
0,transition Frobenius distance,0.575437
1,emission-mean Frobenius distance,4.409426
2,minimum full-pretest occupancy,0.150318
3,maximum classical state duration,3.484211


labels canonicalized by level-factor change mean: True
labels are external observed regimes: False
HMM parameter posterior included: False


## 5. Claim audit

評価可能なのは、固定snapshotと固定horizonにおけるhistorical point/predictive performance、state occupancy/duration、transition/emission stabilityである。stateに「risk-on」「crisis」等の名前を付けるには外部変数・事前定義・再現性検証が必要で、本Projectでは行わない。

HMM intervalはstate/emission randomnessだけでparameter uncertaintyを欠く。Bayesian regressionはfull posterior predictiveだがGaussian linear specificationに依存する。どちらもcoverageとwidthを満たさなければ採用しない。outer testを見た後のstate数変更には新しいholdoutが必要である。

## 6. 失敗モード

- outer testでstate数、prior、featureを選び直す
- smoothed stateをforecast originへ使う
- HMM conditional predictiveをfull Bayesian posterior predictiveと呼ぶ
- labelを市場の真のregimeと呼ぶ
- aggregate coverageだけを報告しtenor failureを隠す
- test log scoreを見てmodel storyを後付けする

## 7. 段階別演習

### 基礎

1. state selection tableとouter evaluationを再現せよ。
2. occupancy、duration、transitionをartifactへ保存せよ。

### 標準

3. horizon 1/20をsecondaryとして追加せよ。
4. filtered probabilityとsmoothed probabilityでpredictive leakage差を示せ。

### 研究

5. Bayesian HMMでtransition/emission parameter uncertaintyを積分する設計を書け。
6. switching Kalman DNSへ拡張する前のsimulation-based calibrationを設計せよ。

## 8. Exit Criteria

- [ ] B7と同じ5公表日targetとouter testを使った
- [ ] state数をtraining/validationだけで固定した
- [ ] forecast originではfiltered probabilityだけを使った
- [ ] random walk、Bayesian、HMMのRMSEを比較した
- [ ] coverage、width、uncertainty contractを分けた
- [ ] occupancy、duration、transition、state-count sensitivityを監査した
- [ ] label switchingとparameter uncertainty不足をclaimへ反映した
- [ ] stateを観測真値・causal regime・PnL signalと呼んでいない

## 9. 出典


- [Gelman et al., Bayesian Data Analysis, 3rd ed.](https://sites.stat.columbia.edu/gelman/book/)
- [Gelman et al., Bayesian Workflow](https://arxiv.org/abs/2011.01808)
- [Vehtari, Gelman, and Gabry, Practical Bayesian model evaluation](https://doi.org/10.1007/s11222-016-9696-4)

- [Rabiner (1989), A Tutorial on Hidden Markov Models](https://www.cs.cmu.edu/~durand/03-711/Readings/Rabiner89.pdf)
- [Vehtari et al., Rank-normalization, folding, and localization](https://arxiv.org/abs/1903.08008)
- [Stan Reference Manual — MCMC Sampling](https://mc-stan.org/docs/reference-manual/mcmc.html)